# AgenticLab Library

## install

In [ ]:
import subprocess
import sys

try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "--quiet"])

try:
    import skimpy
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "skimpy", "--quiet"])

# Desinstala versiones conflictivas
!pip uninstall -y numpy scipy

# Instala NumPy 1.26.4 + SciPy 1.11.3
!pip install --quiet numpy==1.26.4 scipy==1.11.3

import numpy as np, scipy
print("NumPy:", np.__version__)    # → 1.26.4
print("SciPy:", scipy.__version__) # → 1.11.3

import datetime as dt
from pandas.tseries.offsets import BusinessDay
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

#  <center>  Liquidez Diaria – Pipeline AgenticLab </center>
<a class="tocSkip">

# 1. Introducción
<a id='introduction'></a>

### 1.1 Descripción del Problema
Aplicar un “stack” **Engineer → Scientist → Manager** (AgenticLab) para
modelar y pronosticar la liquidez diaria de las carteras *ahorro* y *vista*.

### 1.2 Descripción de los Datos
- **Series históricas**: saldos diarios interpolados de cada cartera.  
- **Calendario**: días hábiles y festivos (banderas 0/1).

Ambos archivos se suministran como CSV.

### 1.3 Descripción del Sistema
La librería se divide en tres capas:

| Capa | Rol | Principal módulo |
|------|-----|------------------|
| 👷 **Engineer** | limpieza, outliers, datasets & transforms | `engineer.py` |
| 👨‍🔬 **Scientist** | HPO, entrenamiento, MC-Dropout | `scientist.py` |
| 👔 **Manager** | orquestación end-to-end | `manager.py` |


# 2. Análisis Exploratorio de Datos
<a id='EDA'></a>


In [ ]:
# ===========================================================
# 1) Imports y config
# ===========================================================
import re, boto3, s3fs, pandas as pd
from urllib.parse import urlparse
from skimpy import skim

BUCKET_NAME   = "ada-us-east-1-sbx-live-mx-m6hn-data"
PATH_CSV_BASE = "data/sandboxes/m6hn/data/coe_liquidez_diaria/csv/"  # termina en /

# ===========================================================
# 2) Detectar última carpeta YYYYMMDD_YYYYMMDD_YYYYMMDD
# ===========================================================
s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

folders = []
for page in paginator.paginate(Bucket=BUCKET_NAME,
                               Prefix=PATH_CSV_BASE,
                               Delimiter="/"):
    for cp in page.get("CommonPrefixes", []):
        folder = cp["Prefix"].replace(PATH_CSV_BASE, "").rstrip("/")
        if folder:
            folders.append(folder)

pattern = re.compile(r'^\d{8}_\d{8}_\d{8}$')
valid_folders = sorted(f for f in folders if pattern.match(f))
if not valid_folders:
    raise RuntimeError("⛔ No hay carpetas válidas en el prefijo")

latest_folder = valid_folders[-1]
print(f"📂 Carpeta más nueva: {latest_folder}")

s3_folder   = f"{PATH_CSV_BASE}{latest_folder}/"
path_csv_diff = f"s3://{BUCKET_NAME}/{s3_folder}full_diff.csv"
path_calendar = (
    "s3://ada-us-east-1-sbx-live-mx-m6hn-data/"
    "data/sandboxes/m6hn/data/coe_liquidez_diaria/"
    "calendario/calendario.csv"
)
print("🔗 diff :", path_csv_diff)
print("🔗 cal  :", path_calendar)

# ===========================================================
# 3) Función robusta para CSV único o carpeta Spark-CSV
# ===========================================================
fs = s3fs.S3FileSystem(anon=False)

def read_csv_s3(path_s3: str, **pd_kw) -> pd.DataFrame:
    """
    Lee un CSV desde S3:
      • Archivo único → s3://bucket/key/file.csv
      • Carpeta Spark-CSV → s3://bucket/key/carpeta.csv/part-*.csv
    Retorna DataFrame concatenado.
    """
    parsed       = urlparse(path_s3)
    bucket, key  = parsed.netloc, parsed.path.lstrip("/")

    # Intentamos listar como carpeta
    try:
        objs = fs.ls(f"{bucket}/{key}")
    except Exception:
        objs = []

    part_files = [o for o in objs if o.endswith(".csv")]
    if part_files:                      # Carpeta Spark-CSV
        df_list = []
        for fk in sorted(part_files):
            with fs.open(fk, "rb") as f:
                df_list.append(pd.read_csv(f, **pd_kw))
        return pd.concat(df_list, ignore_index=True)

    # CSV único
    return pd.read_csv(
        path_s3,
        storage_options={"anon": False},
        **pd_kw
    )

# ===========================================================
# 4) Cargar diff y calendario  (sin parse_dates → convertimos)
# ===========================================================
raw_diff = read_csv_s3(path_csv_diff, low_memory=False)
raw_cal  = read_csv_s3(path_calendar,  low_memory=False)

def normaliza_fecha(df: pd.DataFrame, col_fecha: str = "fecha") -> pd.DataFrame:
    df[col_fecha] = (
        pd.to_datetime(df[col_fecha], errors="coerce")
          .dt.normalize()
    )
    return (
        df.dropna(subset=[col_fecha])
          .set_index(col_fecha)
          .sort_index()
    )

series     = normaliza_fecha(raw_diff.copy())
calendario = normaliza_fecha(raw_cal.copy())

# ===========================================================
# 5) Vista rápida
# ===========================================================
print("\n=== SERIES (df_diff) ===")
skim(series)
print("\n=== CALENDARIO ===")
skim(calendario)

# 3. Tratamiento de Datos (Engineer)
<a id='DW'></a>


In [ ]:
#@title Importar librería AgenticLab

import importlib, sys, os
# Se asume que losses.py, tools.py, engineer.py, scientist.py, manager.py, …
# ya han sido creados anteriormente (Notebook 1).
# Si vienes de cero, ejecuta primero las celdas de definición de módulos.

import engineer as eng
import scientist as sci
import manager   as mng
importlib.reload(eng); importlib.reload(sci); importlib.reload(mng)

from engineer import SystemConfig, Engineer
from scientist import Scientist
from manager   import Manager
from tools     import Tools

#temporal
from sklearn.preprocessing import QuantileTransformer
import torch

import json
import joblib
from pathlib import Path

In [ ]:
#@title Construir Engineer
ALL_COLUMNS = [c for c in series.columns.to_list() if c != "fecha"]

from sklearn.preprocessing import RobustScaler

# Stress-friendly defaults for sparse/signed MAC series.
# full_diff.csv stays in original amounts; Engineer applies signed-log internally
# and Manager inverts predictions back to original amount scale.
TARGET_TRANSFORM = "signed_log"      # @param ["identity", "signed_log"]
MISSING_POLICY   = "ffill_zero"      # @param ["drop", "zero", "ffill_zero", "interpolate_zero"]
FC_HORIZON_DAYS  = 25                # @param {type:"integer"}

engine_cfg = SystemConfig(
    series_columns     = ALL_COLUMNS,
    transformer_cls    = RobustScaler,
    transformer_kwargs = {
        "with_centering": True,
        "with_scaling": True,
        "quantile_range": (25.0, 75.0),
    },
    target_transform   = TARGET_TRANSFORM,
    missing_policy     = MISSING_POLICY,
    min_train_points   = 2,
)

engineer = Engineer(
    series,
    calendario,
    engine_cfg,
    test_size=FC_HORIZON_DAYS,
)

print("\n→ Series crudas       :", engineer.series.shape)
print("→ Calendario          :", engineer.calendar.shape)
print("→ Datasets listos     :", len(engineer.datasets), "/", len(ALL_COLUMNS))
print("→ Target transform    :", TARGET_TRANSFORM)
print("→ Missing policy      :", MISSING_POLICY)
display(engineer.series_status.head(20))


In [ ]:
series.columns.to_list()

In [ ]:
#@title Visualizaciones iniciales de las series originales
PLOT_INPUTS = False  # @param {type:"boolean"}
MAX_PLOT_SERIES = 8  # @param {type:"integer"}

if PLOT_INPUTS:
    engineer.visualize(ALL_COLUMNS[:MAX_PLOT_SERIES])
else:
    print("PLOT_INPUTS=False; se omiten gráficas iniciales para Run All/stress test.")


In [ ]:
#@title Visualizaciones de las series transformadas (y_trans) con outliers
for name in ALL_COLUMNS:
    if name in engineer.datasets:
        ds = engineer.datasets[name]
        print(f"{name:40s} → filas: {ds.shape[0]} | unique_y={ds['y'].nunique(dropna=True)}")

PLOT_TRANSFORMED = False  # @param {type:"boolean"}
if PLOT_TRANSFORMED:
    for name in ALL_COLUMNS[:MAX_PLOT_SERIES]:
        if name not in engineer.datasets:
            continue
        df_t = engineer.datasets[name][["y_trans"]].rename(columns={"y_trans": "total_amount"})
        engineer.tools.plot_series_outliers({name: df_t}, pd.DataFrame(), name)
else:
    print("PLOT_TRANSFORMED=False; se omiten gráficas transformadas para Run All/stress test.")


# 4. División Train/Test y Checks
<a id='traintest'></a>


In [ ]:
#@title Rango back-test / forecast
BT_START = "2025-01-01"

# Usa el máximo índice real de datasets listos, no una lista fija de series.
all_idx = pd.Index([])
for ds in engineer.datasets.values():
    all_idx = all_idx.union(ds.index)

bt_end_dt   = pd.to_datetime(all_idx.max())
bt_end      = bt_end_dt.strftime("%Y-%m-%d")
fc_start_dt = bt_end_dt + dt.timedelta(days=1)
fc_end_dt   = fc_start_dt + dt.timedelta(days=FC_HORIZON_DAYS-1)
fc_start    = fc_start_dt.strftime("%Y-%m-%d")
fc_end      = fc_end_dt.strftime("%Y-%m-%d")

print(f"Back-test : {BT_START} → {bt_end}")
print(f"Forecast  : {fc_start} → {fc_end}  ({FC_HORIZON_DAYS} calendar days)")

series_idx  = all_idx.sort_values()
aligned_idx = engineer.aligned_features.index
print("\n[QC-1] Series   :", series_idx.min(), "→", series_idx.max(), f"({len(series_idx)})")
print("[QC-1] Features :", aligned_idx.min(), "→", aligned_idx.max(), f"({len(aligned_idx)})")

print("✅ Calendario cubre histórico y forecast" if calendario.index.max() >= fc_end_dt
      else f"⚠️ Calendario no cubre todo el forecast – se rellenarán features futuras con 0 hasta {fc_end}")

# Split real por dataset: Manager adapta test_size si alguna serie es corta.
status_split = []
for name, ds in engineer.datasets.items():
    eff_test = min(FC_HORIZON_DAYS, max(1, len(ds)//3)) if len(ds) >= 2 else 0
    status_split.append({"serie": name, "rows": len(ds), "effective_test_size": eff_test})
display(pd.DataFrame(status_split).head(20))


In [ ]:
#@title Visualización de los datos reales con línea Train/Test
PLOT_SPLIT = False  # @param {type:"boolean"}
if PLOT_SPLIT:
    for name in ALL_COLUMNS[:MAX_PLOT_SERIES]:
        if name not in engineer.datasets:
            continue
        ds = engineer.datasets[name]
        split_idx = max(1, len(ds) - min(FC_HORIZON_DAYS, max(1, len(ds)//3)))
        plt.figure(figsize=(14,6))
        plt.plot(ds.index[:split_idx],  ds["y"].values[:split_idx],  marker="o", label="Train")
        plt.plot(ds.index[split_idx:], ds["y"].values[split_idx:], marker="o", label="Test")
        plt.axvline(ds.index[min(split_idx, len(ds)-1)], color="gray", ls="--", label="Train/Test split")
        plt.title(f"{name}: Datos reales con Train/Test")
        plt.xticks(rotation=45); plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("PLOT_SPLIT=False; se omiten gráficas train/test para Run All/stress test.")


# 5. Entrenamiento de Modelos (Scientist)
<a id='Modeling'></a>


In [ ]:
#@title Config generales
# Para smoke/stress test inicial puedes bajar estos valores.
N_MONTE_CARLO = 1000         # @param {type:"integer"}
WARM_TRIALS   = 15           # @param {type:"integer"}
WARM_EPOCHS   = 25           # @param {type:"integer"}
WARM_BATCH    = 32           # @param {type:"integer"}
FINE_EPOCHS   = 35           # @param {type:"integer"}
FINE_BATCH    = 32           # @param {type:"integer"}
LOSS_FUNCTION = "rmse"       # @param ["rmse","mae","mse","smape","wmape","log_cosh","huber"]
SHOW_PLOTS    = False        # @param {type:"boolean"}


In [ ]:
#@title Instanciar Scientist y Manager
scientist = Scientist(
    device="cuda" if torch.cuda.is_available() else "cpu",
    loss_type=LOSS_FUNCTION,
    model_name="mlp",
    suggest_name="mlp",
)

manager = Manager(
    engineer, scientist,
    n_mc      = N_MONTE_CARLO,
    test_size = FC_HORIZON_DAYS,
)


In [ ]:
#@title 5.3 Ejecución paso a paso
print("=== 1/5 Warm-up (Optuna) ===")
manager._warmup_all(
    n_trials=WARM_TRIALS,
    max_epochs=WARM_EPOCHS,
    batch=WARM_BATCH,
    show_progress=True
)

print("\n=== 2/5 Fine-tune ===")
manager._finetune_all(
    epochs=FINE_EPOCHS,
    batch=FINE_BATCH,
    show_progress=True
)

print("\n=== 3/5 Back-test MC-Dropout ===")
manager._run_backtest()

print("\n=== 4/5 Forecast MC-Dropout ===")
manager._run_forecast(
    start_date=fc_start,
    end_date=fc_end
)

print("\n=== 5/5 Visualización ===")
if SHOW_PLOTS:
    manager.visualise(
        bt_start=BT_START,
        bt_end=bt_end,
        fc_start=fc_start,
        fc_end=fc_end
    )
else:
    print("SHOW_PLOTS=False; se omiten visualizaciones finales para Run All/stress test.")

results = {
    "warm":       manager._warmup_results,
    "fine":       manager._finetune_results,
    "backtest":   manager._backtest_results,
    "forecast":   manager._future_results,
    "df_forecasts": manager._df_forecasts,
    "df_outliers":  manager._df_outliers,
    "series_status": manager.series_status,
}

print("\nSeries status:")
display(results["series_status"])


# 6. Métricas de Evaluación
<a id='EVS'></a>


In [ ]:
# @title Métricas robustas
from sklearn.metrics import mean_absolute_error, mean_squared_error, explained_variance_score, r2_score

def metrics_dict(y, f):
    return Tools.metrics_regression(np.asarray(y), np.asarray(f))

def build_metrics(df_subset):
    rows = []
    if df_subset is None or df_subset.empty:
        return pd.DataFrame()
    for serie, grp in df_subset.groupby("serie"):
        m = metrics_dict(grp["actual_orig"].values, grp["pred_orig"].values)
        m["serie"] = serie
        if "bias2" in grp.columns:
            m["Bias2_mean"] = float(grp["bias2"].mean())
        if "var_pred" in grp.columns:
            m["Var_mean"] = float(grp["var_pred"].mean())
        rows.append(m)
    if not rows:
        return pd.DataFrame()
    rename = {"MAPE":"MAPE (%)", "sMAPE":"sMAPE (%)", "WMAPE":"WMAPE (%)", "MedAPE":"MedAPE (%)"}
    return pd.DataFrame(rows).set_index("serie").rename(columns=rename).sort_index()


In [ ]:
#@title Warm-up
df_warm = results["warm"].get("df_regression", pd.DataFrame())
print("\n============= MÉTRICAS DE REGRESIÓN (Warm-up) =============")
display(build_metrics(df_warm))


In [ ]:
#@title Fine-tune
df_fine = results["fine"].get("df_regression", pd.DataFrame())
print("\n============= MÉTRICAS DE REGRESIÓN (Fine-tune) =============")
display(build_metrics(df_fine))


In [ ]:
#@title Back-test (train / test)
df_bt = results["backtest"].get("df_regression", pd.DataFrame())

print("\n============= TRAIN BACK-TEST =============")
train_metrics = build_metrics(df_bt[df_bt["isTrain"].astype(bool)] if not df_bt.empty and "isTrain" in df_bt else pd.DataFrame())
display(train_metrics)

print("\n============= TEST BACK-TEST =============")
test_metrics = build_metrics(df_bt[~df_bt["isTrain"].astype(bool)] if not df_bt.empty and "isTrain" in df_bt else pd.DataFrame())
display(test_metrics)


# 8. Escenarios Proyectados
<a id='PEI'></a>


In [ ]:
#@title Forecast
results["df_forecasts"]

# 9. Exportación de Artefactos y Logs


In [ ]:
# ╭─────────────────────────────────────────────────────────────╮
# │ 0) Imports necesarios                                       │
# ╰─────────────────────────────────────────────────────────────╯
import json, joblib, io, datetime as dt, boto3, re
import pandas as pd
import numpy as np
from pathlib import Path

# ╭─────────────────────────────────────────────────────────────╮
# │ 1) Parámetros de ejecución                                  │
# ╰─────────────────────────────────────────────────────────────╯
ALGO   = "MLP_E_D"
TODAY  = dt.datetime.now().strftime("%Y%m%d")
RUN_TS = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
BUCKET = "ada-us-east-1-sbx-live-mx-m6hn-data"
PREFIX = f"data/sandboxes/m6hn/data/coe_liquidez_diaria/execution/{TODAY}/{ALGO}_{RUN_TS}/"

s3 = boto3.client("s3")

def safe_artifact_name(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("._-") or "serie"

def to_s3_csv(df, name):
    buffer = io.StringIO()
    df.to_csv(buffer, index=True)
    s3.put_object(Bucket=BUCKET, Key=PREFIX + f"data/{name}", Body=buffer.getvalue())

def to_s3_json(obj, name):
    def _default(o):
        if isinstance(o, (np.integer, np.floating)):
            return o.item()
        if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
            return str(o)
        return str(o)
    s3.put_object(Bucket=BUCKET, Key=PREFIX + f"utils/{name}", Body=json.dumps(obj, indent=2, default=_default))

def to_s3_joblib(obj, name):
    buffer = io.BytesIO()
    joblib.dump(obj, buffer)
    buffer.seek(0)
    s3.upload_fileobj(buffer, BUCKET, PREFIX + f"utils/{name}")

def calc_metrics_df(df):
    return build_metrics(df)

def save_metrics(df, filename):
    to_s3_csv(calc_metrics_df(df), filename)

# Datos base
to_s3_csv(series, "series_raw.csv")
to_s3_csv(calendario, "calendario_raw.csv")
to_s3_csv(engineer.aligned_features, "aligned_features.csv")
to_s3_csv(engineer.outliers, "outliers.csv")
to_s3_csv(engineer.series_status, "series_status_engineer.csv")
to_s3_csv(manager.series_status, "series_status_manager.csv")

# Resultados pipeline
df_bt_all = results["backtest"].get("df_regression", pd.DataFrame())
to_s3_csv(df_bt_all, "backtest_timeseries.csv")
to_s3_csv(results.get("df_forecasts", pd.DataFrame()), "forecast_timeseries.csv")

# Datasets por serie, con nombre seguro para filesystem/S3
manifest_rows = []
for serie, df in engineer.datasets.items():
    safe = safe_artifact_name(serie)
    filename = f"dataset_{safe}.csv"
    to_s3_csv(df, filename)
    manifest_rows.append({"serie": serie, "safe_name": safe, "filename": filename})
to_s3_csv(pd.DataFrame(manifest_rows), "dataset_manifest.csv")

# Métricas
save_metrics(results["warm"].get("df_regression", pd.DataFrame()),  "metrics_warm_up.csv")
save_metrics(results["fine"].get("df_regression", pd.DataFrame()),  "metrics_fine_tune.csv")

if df_bt_all.empty:
    print("[WARN] df_bt_all está vacío; no se exportan métricas backtest.")
else:
    if "isTrain" not in df_bt_all.columns:
        raise KeyError("df_bt_all no trae columna 'isTrain' (requerida para train/test).")
    is_train = df_bt_all["isTrain"].astype(bool)
    for subset, mask in {"train": is_train, "test": ~is_train}.items():
        df_metrics = calc_metrics_df(df_bt_all[mask].copy())
        to_s3_csv(df_metrics, f"metrics_backtest_{subset}.csv")
        if subset == "test":
            to_s3_csv(df_metrics, "backtest_metrics.csv")

# Logs/config
to_s3_json({
    "ALGO": ALGO,
    "TARGET_TRANSFORM": TARGET_TRANSFORM,
    "MISSING_POLICY": MISSING_POLICY,
    "FC_HORIZON_DAYS": FC_HORIZON_DAYS,
    "N_MONTE_CARLO": N_MONTE_CARLO,
    "WARM_TRIALS": WARM_TRIALS,
    "WARM_EPOCHS": WARM_EPOCHS,
    "FINE_EPOCHS": FINE_EPOCHS,
    "LOSS_FUNCTION": LOSS_FUNCTION,
    "series_count": len(ALL_COLUMNS),
}, "run_config.json")
to_s3_joblib(engineer.transforms, "target_transforms.joblib")

def show_log(title, checklist):
    print(f"\n=== {title} ({len(checklist)}) ===")
    for i, step in enumerate(checklist, 1):
        print(f"{i:2d}. {step}")

show_log("ENGINEER", engineer.checklist)
show_log("SCIENTIST", scientist.checklist)
show_log("MANAGER", manager.checklist)

print(f"\n✓ Artefactos subidos en: s3://{BUCKET}/{PREFIX}")
